# **FEATURE ENGINEERING NOTEBOOK**

## Objectives

*   Engineer features for Classification model


## Inputs

* **Raw Dataset:** inputs/datasets/raw/hotel_bookings.csv

## Outputs

* Summary of feature engineering steps to try
* Code for feature engineering pipeline


---

# Imports

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

# Load, Pre-Process, Split and Clean Data

## Load Data

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
dataset_file = project_root / 'inputs' / 'datasets' / 'raw' / 'hotel_bookings.csv'
df = pd.read_csv(
    dataset_file,
    dtype={
        'agent': 'object',
        'company': 'object',
        'is_repeated_guest': 'object',
    },
)
print('Shape:', df.shape)
df.head(3)

## Pre-Split Data Cleaning

Then drop columns `reservation_status` and `reservation_status_date`

In [ ]:
df = df.drop(columns=['reservation_status', 'reservation_status_date'])
print('Shape:', df.shape)
df.head(3)

Remove duplicates

In [ ]:
df = df.value_counts(dropna=False).reset_index(name='record_count')
df['is_duplicate'] = (df['record_count'] > 1).astype('int')
print('Shape:', df.shape)
df.head(3)

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('is_canceled', axis=1), df['is_canceled'], test_size=0.2, random_state=0)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train.head(3)

## Data Cleaning Pipeline

Define custom transformer for imputing `is_repeated_guest`

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class RepeatedGuestImputer(BaseEstimator, TransformerMixin):
    def __init__(self, booking_col='previous_bookings_not_canceled', target_col='is_repeated_guest'):
        self.booking_col = booking_col
        self.target_col = target_col

    def fit(self, X, y=None):
        # Set a dummy attribute to signal fitted status (to remove warning)
        self.fitted_ = True
        return self

    def transform(self, X):
        X = X.copy()
        mask_missing = X[self.target_col].isnull()
        X.loc[mask_missing, self.target_col] = (X.loc[mask_missing, self.booking_col] > 0).astype(int)
        return X


Define variables and transformers for using in the pipeline

In [ ]:
from feature_engine.imputation import (
    CategoricalImputer,
    ArbitraryNumberImputer,
    MeanMedianImputer,
)
from feature_engine.outliers import ArbitraryOutlierCapper

# Categorical imputer for 'Missing' label
impute_missing_label_variables = ['country', 'company', 'agent']
imputer_missing_label = CategoricalImputer(
    imputation_method='missing', 
    fill_value='Missing',
    variables=impute_missing_label_variables,
)

# Categorical imputer for mode
impute_mode_variables = [
    'hotel', 'arrival_date_month', 'meal', 'market_segment',
    'distribution_channel', 'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'customer_type'
]
imputer_mode = CategoricalImputer(
    imputation_method='frequent',
    variables=impute_mode_variables
)

# Categorical imputer for repeated guest
imputer_repeated_guest = RepeatedGuestImputer()

# Numeric imputer for zero
impute_zero_variables = [
    'children', 'babies', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'required_car_parking_spaces', 'total_of_special_requests'
]
imputer_zero = ArbitraryNumberImputer(
    arbitrary_number=0, 
    variables=impute_zero_variables
)

# Numeric imputer for median
impute_median_variables = ['adr', 'adults']
imputer_median = MeanMedianImputer(
    imputation_method='median',
    variables=impute_median_variables
)

# Outlier capper
max_capping_dict = {
    'lead_time': 600,
    'arrival_date_year': 2017,
    'arrival_date_week_number': 53,
    'arrival_date_day_of_month': 31,
    'stays_in_weekend_nights': 10,
    'stays_in_week_nights': 25,
    'adults': 4,
    'children': 4,
    'babies': 2,
    'previous_cancellations': 10,
    'previous_bookings_not_canceled': 20,
    'booking_changes': 10,
    'days_in_waiting_list': 60,
    'adr': 400,
    'required_car_parking_spaces': 2,
    'total_of_special_requests': 5
}

min_capping_dict = {
    'lead_time': 1,
    'arrival_date_year': 2015,
    'arrival_date_week_number': 1,
    'arrival_date_day_of_month': 1,
    'stays_in_weekend_nights': 0,
    'stays_in_week_nights': 0,
    'adults': 1,
    'children': 0,
    'babies': 0,
    'previous_cancellations': 0,
    'previous_bookings_not_canceled': 0,
    'booking_changes': 0,
    'days_in_waiting_list': 0,
    'adr': 0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 0
}

outlier_capper = ArbitraryOutlierCapper(
    max_capping_dict=max_capping_dict,
    min_capping_dict=min_capping_dict
)

Define data cleaning pipeline

In [ ]:
from sklearn.pipeline import Pipeline

data_cleaning_pipeline = Pipeline([
    ('imputer_missing_label', imputer_missing_label),
    ('imputer_mode', imputer_mode),
    ('imputer_repeated_guest', imputer_repeated_guest),
    ('imputer_zero', imputer_zero),
    ('imputer_median', imputer_median),
    ('outlier_capper', outlier_capper),
])

Fit the pipeline and transform data.

In [ ]:
X_train = data_cleaning_pipeline.fit_transform(X_train)
X_test = data_cleaning_pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

Copy for testing the feature engineering pipeline at the end of the notebook

In [ ]:
X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

---

# Feature Engineering

## Custom function

This function (written by Code Institute) and will be used to assist in the analysis that follows.

In [ ]:
import scipy.stats as stats
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
from feature_engine import transformation as vt
from feature_engine.outliers import Winsorizer
from feature_engine.encoding import OrdinalEncoder
sns.set(style="whitegrid")
warnings.filterwarnings('ignore')


def FeatureEngineeringAnalysis(df, analysis_type=None):
    """
    - used for quick feature engineering on numerical and categorical variables
    to decide which transformation can better transform the distribution shape
    - Once transformed, use a reporting tool, like ydata-profiling, to evaluate distributions
    """
    check_missing_values(df)
    allowed_types = ['numerical', 'ordinal_encoder', 'outlier_winsorizer']
    check_user_entry_on_analysis_type(analysis_type, allowed_types)
    list_column_transformers = define_list_column_transformers(analysis_type)

    # Loop in each variable and engineer the data according to the analysis type
    df_feat_eng = pd.DataFrame([])
    for column in df.columns:
        # create additional columns (column_method) to apply the methods
        df_feat_eng = pd.concat([df_feat_eng, df[column]], axis=1)
        for method in list_column_transformers:
            df_feat_eng[f"{column}_{method}"] = df[column]

        # Apply transformers in respective column_transformers
        df_feat_eng, list_applied_transformers = apply_transformers(
            analysis_type, df_feat_eng, column)

        # For each variable, assess how the transformations perform
        transformer_evaluation(
            column, list_applied_transformers, analysis_type, df_feat_eng)

    return df_feat_eng


def check_user_entry_on_analysis_type(analysis_type, allowed_types):
    """ Check analysis type """
    if analysis_type is None:
        raise SystemExit(
            f"You should pass analysis_type parameter as one of the following options: {allowed_types}")
    if analysis_type not in allowed_types:
        raise SystemExit(
            f"analysis_type argument should be one of these options: {allowed_types}")


def check_missing_values(df):
    if df.isna().sum().sum() != 0:
        raise SystemExit(
            f"There is a missing value in your dataset. Please handle that before getting into feature engineering.")


def define_list_column_transformers(analysis_type):
    """ Set suffix columns according to analysis_type"""
    if analysis_type == 'numerical':
        list_column_transformers = [
            "log_e", "log_10", "reciprocal", "power", "box_cox", "yeo_johnson"]

    elif analysis_type == 'ordinal_encoder':
        list_column_transformers = ["ordinal_encoder"]

    elif analysis_type == 'outlier_winsorizer':
        list_column_transformers = ['iqr']

    return list_column_transformers


def apply_transformers(analysis_type, df_feat_eng, column):
    for col in df_feat_eng.select_dtypes(include='category').columns:
        df_feat_eng[col] = df_feat_eng[col].astype('object')

    if analysis_type == 'numerical':
        df_feat_eng, list_applied_transformers = FeatEngineering_Numerical(
            df_feat_eng, column)

    elif analysis_type == 'outlier_winsorizer':
        df_feat_eng, list_applied_transformers = FeatEngineering_OutlierWinsorizer(
            df_feat_eng, column)

    elif analysis_type == 'ordinal_encoder':
        df_feat_eng, list_applied_transformers = FeatEngineering_CategoricalEncoder(
            df_feat_eng, column)

    return df_feat_eng, list_applied_transformers


def transformer_evaluation(column, list_applied_transformers, analysis_type, df_feat_eng):
    # For each variable, assess how the transformations perform
    print(f"* Variable Analyzed: {column}")
    print(f"* Applied transformation: {list_applied_transformers} \n")
    for col in [column] + list_applied_transformers:

        if analysis_type != 'ordinal_encoder':
            DiagnosticPlots_Numerical(df_feat_eng, col)

        else:
            if col == column:
                DiagnosticPlots_Categories(df_feat_eng, col)
            else:
                DiagnosticPlots_Numerical(df_feat_eng, col)

        print("\n")


def DiagnosticPlots_Categories(df_feat_eng, col):
    plt.figure(figsize=(4, 3))
    sns.countplot(data=df_feat_eng, x=col, palette=[
                  '#432371'], order=df_feat_eng[col].value_counts().index)
    plt.xticks(rotation=90)
    plt.suptitle(f"{col}", fontsize=30, y=1.05)
    plt.show()
    print("\n")


def DiagnosticPlots_Numerical(df, variable):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    sns.histplot(data=df, x=variable, kde=True, element="step", ax=axes[0])
    stats.probplot(df[variable], dist="norm", plot=axes[1])
    sns.boxplot(x=df[variable], ax=axes[2])

    axes[0].set_title('Histogram')
    axes[1].set_title('QQ Plot')
    axes[2].set_title('Boxplot')
    fig.suptitle(f"{variable}", fontsize=30, y=1.05)
    plt.tight_layout()
    plt.show()


def FeatEngineering_CategoricalEncoder(df_feat_eng, column):
    list_methods_worked = []
    try:
        encoder = OrdinalEncoder(encoding_method='arbitrary', variables=[
                                 f"{column}_ordinal_encoder"])
        df_feat_eng = encoder.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_ordinal_encoder")

    except Exception:
        df_feat_eng.drop([f"{column}_ordinal_encoder"], axis=1, inplace=True)

    return df_feat_eng, list_methods_worked


def FeatEngineering_OutlierWinsorizer(df_feat_eng, column):
    list_methods_worked = []

    # Winsorizer iqr
    try:
        disc = Winsorizer(
            capping_method='iqr', tail='both', fold=1.5, variables=[f"{column}_iqr"])
        df_feat_eng = disc.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_iqr")
    except Exception:
        df_feat_eng.drop([f"{column}_iqr"], axis=1, inplace=True)

    return df_feat_eng, list_methods_worked


def FeatEngineering_Numerical(df_feat_eng, column):
    list_methods_worked = []

    # LogTransformer base e
    try:
        lt = vt.LogTransformer(variables=[f"{column}_log_e"])
        df_feat_eng = lt.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_log_e")
    except Exception:
        df_feat_eng.drop([f"{column}_log_e"], axis=1, inplace=True)

    # LogTransformer base 10
    try:
        lt = vt.LogTransformer(variables=[f"{column}_log_10"], base='10')
        df_feat_eng = lt.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_log_10")
    except Exception:
        df_feat_eng.drop([f"{column}_log_10"], axis=1, inplace=True)

    # ReciprocalTransformer
    try:
        rt = vt.ReciprocalTransformer(variables=[f"{column}_reciprocal"])
        df_feat_eng = rt.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_reciprocal")
    except Exception:
        df_feat_eng.drop([f"{column}_reciprocal"], axis=1, inplace=True)

    # PowerTransformer
    try:
        pt = vt.PowerTransformer(variables=[f"{column}_power"])
        df_feat_eng = pt.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_power")
    except Exception:
        df_feat_eng.drop([f"{column}_power"], axis=1, inplace=True)

    # BoxCoxTransformer
    try:
        bct = vt.BoxCoxTransformer(variables=[f"{column}_box_cox"])
        df_feat_eng = bct.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_box_cox")
    except Exception:
        df_feat_eng.drop([f"{column}_box_cox"], axis=1, inplace=True)

    # YeoJohnsonTransformer
    try:
        yjt = vt.YeoJohnsonTransformer(variables=[f"{column}_yeo_johnson"])
        df_feat_eng = yjt.fit_transform(df_feat_eng)
        list_methods_worked.append(f"{column}_yeo_johnson")
    except Exception:
        df_feat_eng.drop([f"{column}_yeo_johnson"], axis=1, inplace=True)

    return df_feat_eng, list_methods_worked


## Encoding Binary Categorical Features

### Inspect

View data types

In [ ]:
X_train.info()

View binary categorical features

In [ ]:
# Get all categorical features as a list
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.to_list()

# Get low cardinality categorical features
binary_cats = [col for col in cat_cols if (X_train[col].nunique() <= 2)]
print('Binary categorical features:')
display(binary_cats)

The only binary categorical feature was `hotel`. This feature is suitable for ordinal encoding.

### Ordinal Encoding

Make a copy of the dataframe with just the variable to apply ordinal encoding to

In [ ]:
binary_variables = binary_cats.copy()

df_engineering = X_train[binary_variables].copy()
df_engineering.head(3)

Test ordinal encoder on df_engineering

In [ ]:
from feature_engine.encoding import OrdinalEncoder

ord_enc = OrdinalEncoder(encoding_method='arbitrary', variables=binary_variables)

df_engineering = ord_enc.fit_transform(df_engineering)
df_engineering.head(3)

The transformer worked correctly and will therefore be applied to the train and test sets

In [ ]:
X_train = ord_enc.fit_transform(X_train)
X_test = ord_enc.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

## Encoding Low Cardinality Categorical Features

### Inspect

View data types

In [ ]:
X_train.info()

View low cardinality categorical features

In [ ]:
# Get all categorical features as a list
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.to_list()

# Get low cardinality categorical features
low_card_cats = [col for col in cat_cols if (3 <= X_train[col].nunique() < 15)]
print('Low cardinality features:')
display(low_card_cats)

All are suitable for one-hot encoding but it may be better to apply cyclical encoding with `arrival_date_month` so the model understands that December and January are close to each other. This could be done using sin and cos functions.

### One-Hot Encoding

Make a copy of dataframe with just the variables that will be one hot encoded

In [ ]:
one_hot_cats = [col for col in low_card_cats if col != 'arrival_date_month']

df_engineering = X_train[one_hot_cats].copy()
df_engineering.head(3)

Test one-hot encoder on df_engineering

In [ ]:
from feature_engine.encoding import OneHotEncoder

ohe = OneHotEncoder(variables=one_hot_cats, drop_last=False)
df_engineering = ohe.fit_transform(df_engineering)

df_engineering.head(3)

The transformer worked correctly and will therefore be applied to the train and test sets

In [ ]:
X_train = ohe.fit_transform(X_train)
X_test = ohe.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

### Cyclical Encoding

Define custom transformer

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class MonthMapper(BaseEstimator, TransformerMixin):
    def __init__(self, variables):
        self.variables = variables
        self.month_map = {
            'January': 1, 'February': 2, 'March': 3, 'April': 4,
            'May': 5, 'June': 6, 'July': 7, 'August': 8,
            'September': 9, 'October': 10, 'November': 11, 'December': 12
        }
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[var].map(self.month_map)
        return X

Make a copy of dataframe with just the variable that will be transformed

In [ ]:
month_variables = ['arrival_date_month']

df_engineering = X_train[month_variables].copy()
df_engineering.head(3)

Fit MonthMapper transformer to df_engineering

In [ ]:
month_mapper = MonthMapper(variables=month_variables)
df_engineering = month_mapper.fit_transform(df_engineering)

df_engineering.head(3)

This transformer worked correctly. Now apply CyclicalFeatures to df_engineering

In [ ]:
from feature_engine.creation import CyclicalFeatures

cf = CyclicalFeatures(variables=month_variables, drop_original=True)
df_engineering = cf.fit_transform(df_engineering)

df_engineering.head(3)

This transformer also worked correctly. Build a pipeline to apply both transformers to the train and test sets.

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('month_mapper', MonthMapper(variables=month_variables)),
    ('month_cyclical', CyclicalFeatures(variables=month_variables))
])

X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

This pipeline worked correctly

## Encoding High Cardinality Categorical Features

### Inspect

In [ ]:
X_train.info()

In [ ]:
# Get all categorical features as a list
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.to_list()

# Get high cardinality categorical features and drop from dataframe
high_card_cats = [col for col in cat_cols if X_train[col].nunique() >= 15]
print('\nHigh cardinality features:\n', high_card_cats)

Use custom function from Notebook 2 to assess the categories that are very rare

In [ ]:
def value_counts_and_percentages(df, filter_by_cols=None):
    data = df[filter_by_cols] if filter_by_cols else df
    df_count = data.value_counts(dropna=False)
    df_percent = round(data.value_counts(normalize=True, dropna=False) * 100, 1)
    summary = pd.concat([df_count, df_percent], axis=1)
    summary.columns = ['Count', '%']
    return summary

### Country - RareLabelEncoder then OneHotEncoder

Make a copy of the dataframe with just the variable that will be transformed

In [ ]:
variables_engineering = ['country']

df_engineering = X_train[variables_engineering].copy()
df_engineering.head(3)

View proportions of different classes to determine the percentage at which classes are considered rare

In [ ]:
summary = value_counts_and_percentages(df=df_engineering, filter_by_cols=variables_engineering)
summary.head(20).T

Earlier analysis revealed that some of the countries with smaller proportions had among the highest high percentage cancellations (e.g. China and Russia), so perhaps a threshold of 0.5% would be a good starting point for investigations.

Test RareLabelEncoder on df_engineering

In [ ]:
from feature_engine.encoding import RareLabelEncoder

# Group categories with <0.5% of the data
rare_enc = RareLabelEncoder(tol=0.005, n_categories=1, variables=variables_engineering)
df_engineering = rare_enc.fit_transform(df_engineering)

summary = value_counts_and_percentages(df=df_engineering, filter_by_cols=variables_engineering)
summary.T

The RareLabelEncoder worked correctly. Now apply one-hot encoding.

In [ ]:
ohe = OneHotEncoder(variables=variables_engineering, drop_last=False)
df_engineering = ohe.fit_transform(df_engineering)

df_engineering.head(3)

This transformer also worked correctly. Build a pipeline to apply both transformers to the train and test sets.

In [ ]:
pipeline = Pipeline([
    ('rare_enc', RareLabelEncoder(tol=0.005, n_categories=1, variables=variables_engineering)),
    ('one_hot_enc', OneHotEncoder(variables=variables_engineering, drop_last=False))
])

X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

This pipeline worked correctly

### Agent - RareLabelEncoder then OneHotEncoder

Make a copy of the dataframe with just the variable that will be transformed

In [ ]:
variables_engineering = ['agent']

df_engineering = X_train[variables_engineering].copy()
df_engineering.head(3)

View proportions of different classes to determine the percentage at which classes are considered rare

In [ ]:
summary = value_counts_and_percentages(df=df_engineering, filter_by_cols=variables_engineering)
summary.head(20).T

Earlier analysis revealed that agents with the highest percentage cancellations included 9, 240, 1, 242 and 8, so perhaps a threshold of 0.5% would be a good starting point for investigations.

Test RareLabelEncoder on df_engineering

In [ ]:
from feature_engine.encoding import RareLabelEncoder

# Group categories with <0.5% of the data
rare_enc = RareLabelEncoder(tol=0.005, n_categories=1, variables=variables_engineering)
df_engineering = rare_enc.fit_transform(df_engineering)

summary = value_counts_and_percentages(df=df_engineering, filter_by_cols=variables_engineering)
summary.T

The RareLabelEncoder worked correctly. Now apply one-hot encoding.

In [ ]:
ohe = OneHotEncoder(variables=variables_engineering, drop_last=False)
df_engineering = ohe.fit_transform(df_engineering)

df_engineering.head(3)

This transformer also worked correctly. Build a pipeline to apply both transformers to the train and test sets.

In [ ]:
pipeline = Pipeline([
    ('rare_enc', RareLabelEncoder(tol=0.005, n_categories=1, variables=variables_engineering)),
    ('one_hot_enc', OneHotEncoder(variables=variables_engineering, drop_last=False))
])

X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

This pipeline worked correctly

### Company - RareLabelEncoder then OneHotEncoder

Make a copy of the dataframe with just the variable that will be transformed

In [ ]:
variables_engineering = ['company']

df_engineering = X_train[variables_engineering].copy()
df_engineering.head(3)

View proportions of different classes to determine the percentage at which classes are considered rare

In [ ]:
summary = value_counts_and_percentages(df=df_engineering, filter_by_cols=variables_engineering)
summary.head(30).T

In the vast majority of records, no company is specified. A threshold of 1% would keep the two dominant categories and prevent the model from overfitting on tiny categories.

Test RareLabelEncoder on df_engineering

In [ ]:
from feature_engine.encoding import RareLabelEncoder

# Group categories with <0.5% of the data
rare_enc = RareLabelEncoder(tol=0.01, n_categories=1, variables=variables_engineering)
df_engineering = rare_enc.fit_transform(df_engineering)

summary = value_counts_and_percentages(df=df_engineering, filter_by_cols=variables_engineering)
summary.T

The RareLabelEncoder worked correctly. Now apply one-hot encoding.

In [ ]:
ohe = OneHotEncoder(variables=variables_engineering, drop_last=False)
df_engineering = ohe.fit_transform(df_engineering)

df_engineering.head(3)

This transformer also worked correctly. Build a pipeline to apply both transformers to the train and test sets.

In [ ]:
pipeline = Pipeline([
    ('rare_enc', RareLabelEncoder(tol=0.01, n_categories=1, variables=variables_engineering)),
    ('one_hot_enc', OneHotEncoder(variables=variables_engineering, drop_last=False))
])

X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

This pipeline worked correctly

## Encoding Other Cyclical Features

In [ ]:
df.select_dtypes(include='number').info()

### Week number and Day of Month 

`arrival_date_week_number` and `arrival_date_day_of_month` are cyclical in nature.

These can be encoded using `CyclicalEncoder` - the custom transformer used earlier.

Make a copy of dataframe with `arrival_date_week_number` and `arrival_date_day_of_month` as the only variables

In [ ]:
cyclical_variables = ['arrival_date_week_number', 'arrival_date_day_of_month']

df_engineering = X_train[cyclical_variables].copy()
df_engineering.head(3)

Fit transformer to df_engineering - maximum values will be determined during fitting

In [ ]:
cf = CyclicalFeatures(variables=cyclical_variables, drop_original=True)
df_engineering = cf.fit_transform(df_engineering)

df_engineering.head(3)

The transformer worked correctly. Apply the transformer to the train and test sets.

In [ ]:
X_train = cf.fit_transform(X_train)
X_test = cf.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

## Feature Creation Using Numeric Features

### Total Stay Length

It could be useful having a feature which summarises the total length of stay. This is easily calculated as `stays_in_weekend_nights` + `stays_in_week_nights`.

Make a copy of the dataframe including `stays_in_weekend_nights` and `stays_in_week_nights` as the only variables

In [ ]:
add_variables = ['stays_in_weekend_nights', 'stays_in_week_nights']

df_engineering = X_train[add_variables].copy()
df_engineering.head(3)

Fit transformer to df_engineering

In [ ]:
from feature_engine.creation import MathFeatures

total_stays = MathFeatures(variables=add_variables, func='sum', new_variables_names=['total_stay_length'])
df_engineering = total_stays.fit_transform(df_engineering)

df_engineering.head(3)

The transformer worked correctly. Apply the transformer to the train and test sets.

In [ ]:
X_train = total_stays.fit_transform(X_train)
X_test = total_stays.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

### Total Cost

It could be useful having a feature which summarises the total cost of the stay. This is easily calculated as `adr` × `total_stay_length`.

Make a copy of the dataframe including `adr` and `total_stay_length` as the only variables

In [ ]:
mult_variables = ['adr', 'total_stay_length']

df_engineering = X_train[mult_variables].copy()
df_engineering.head(3)

Fit transformer to df_engineering

In [ ]:
from feature_engine.creation import MathFeatures

total_cost = MathFeatures(variables=mult_variables, func='prod', new_variables_names=['total_cost'])
df_engineering = total_cost.fit_transform(df_engineering)

df_engineering.head(3)

The transformer worked correctly. Apply the transformer to the train and test sets.

In [ ]:
X_train = total_cost.fit_transform(X_train)
X_test = total_cost.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

### Cancellation Ratio

It could be useful having a feature which summarises the cancellation ratio:
- cancels / previous total bookings.

A custom function is needed for this

In [ ]:
class CancellationRatio(BaseEstimator, TransformerMixin):
    """
    Creates cancellation_ratio = previous_cancellations / (previous_total_bookings)
    """

    def __init__(self,
                 cancel_col='previous_cancellations',
                 no_cancel_col='previous_bookings_not_canceled',
                 new_col_name='cancellation_ratio'):
        self.cancel_col = cancel_col
        self.no_cancel_col = no_cancel_col
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        total_prev = X[self.cancel_col] + X[self.no_cancel_col]
        # Avoid division by zero
        X[self.new_col_name] = np.where(total_prev == 0, 0, X[self.cancel_col] / total_prev)
        return X


Make a copy of the dataframe including `previous_cancellations` and `previous_bookings_not_canceled` as the only variables

In [ ]:
feature_variables = ['previous_cancellations', 'previous_bookings_not_canceled']

df_engineering = X_train[feature_variables].copy()
df_engineering.head(3)

Fit transformer to df_engineering

In [ ]:
cancel_ratio = CancellationRatio()
df_engineering = cancel_ratio.fit_transform(df_engineering)

df_engineering.head(3)

Check transformer has worked

In [ ]:
print('PREVIOUS NOT CANCELLED:')
display(df_engineering.query("previous_bookings_not_canceled > 0").head(2))
print('\nPREVIOUS CANCELLED:')
display(df_engineering.query("previous_cancellations > 0").head(2))

The transformer worked correctly. Apply the transformer to the train and test sets.

In [ ]:
X_train = cancel_ratio.fit_transform(X_train)
X_test = cancel_ratio.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

## Numeric Distributions

### Average Daily Rate and Lead Time

Make a copy of the dataframe with only `adr` and `lead_time`

In [ ]:
variables_engineering = ['adr', 'lead_time']

df_engineering = X_train[variables_engineering].copy()
df_engineering.head(3)

### Winsorization

Assess Winsorization (cap outliers beyond 1.5 × IQR)

In [ ]:
summary = FeatureEngineeringAnalysis(df=df_engineering, analysis_type='outlier_winsorizer')

These are not necessary and have a negative effect on the normalisation of distributions so will not be included in the feature engineering pipeline.

### Normalising Data

Assess numerical distributions for `adr` and `lead_time`

In [ ]:
summary = FeatureEngineeringAnalysis(df=df_engineering, analysis_type='numerical')

For `adr`, the Yeo-Johnson transformation best improved the normalisation of the data.

For `lead_time`, the Power transformation best improved the normalisation of the data.

Fit the Yeo-Johnson transformer for `adr`

In [ ]:
from feature_engine.transformation import YeoJohnsonTransformer

yjt = YeoJohnsonTransformer(variables=['adr'])
df_engineering = yjt.fit_transform(df_engineering)

df_engineering.head(3)

The transformer worked correctly. Apply the transformer to the train and test sets.

In [ ]:
X_train = yjt.fit_transform(X_train)
X_test = yjt.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

Fit the PowerTransformer for `lead_time`

In [ ]:
from feature_engine.transformation import PowerTransformer

pt = PowerTransformer(variables=['lead_time'])
df_engineering = pt.fit_transform(df_engineering)

df_engineering.head(3)

The power transformer worked correctly. Apply the transformer to the train and test sets.

In [ ]:
X_train = pt.fit_transform(X_train)
X_test = pt.transform(X_test)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

X_train.head(3)

## Smart Correlated Selection

Make a copy of the dataframe with all variables`

In [ ]:
df_engineering = X_train.copy()
df_engineering.head(3)

Fit SmartCorrelatedSelection to df_engineering

In [ ]:
from feature_engine.selection import SmartCorrelatedSelection

corr_sel = SmartCorrelatedSelection(variables=None, method='spearman', threshold=0.6, selection_method='variance')

corr_sel.fit_transform(df_engineering)
corr_sel.correlated_feature_sets_

See which features are recommended to be dropped

In [ ]:
corr_sel.features_to_drop_

---

# Summary of Feature Engineering Pipeline

Define custom transformers

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class CancellationRatio(BaseEstimator, TransformerMixin):
    """
    Creates cancellation_ratio = previous_cancellations / (previous_total_bookings)
    """

    def __init__(self,
                 cancel_col='previous_cancellations',
                 no_cancel_col='previous_bookings_not_canceled',
                 new_col_name='cancellation_ratio'):
        self.cancel_col = cancel_col
        self.no_cancel_col = no_cancel_col
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        total_prev = X[self.cancel_col] + X[self.no_cancel_col]
        # Avoid division by zero
        X[self.new_col_name] = np.where(total_prev == 0, 0, X[self.cancel_col] / total_prev)
        return X


class MonthMapper(BaseEstimator, TransformerMixin):
    def __init__(self, variables):
        self.variables = variables
        self.month_map = {
            'January': 1, 'February': 2, 'March': 3, 'April': 4,
            'May': 5, 'June': 6, 'July': 7, 'August': 8,
            'September': 9, 'October': 10, 'November': 11, 'December': 12
        }
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[var].map(self.month_map)
        return X

Define variables and transformers

In [ ]:
from feature_engine.encoding import OrdinalEncoder, OneHotEncoder, RareLabelEncoder
from feature_engine.creation import CyclicalFeatures, MathFeatures
from feature_engine.transformation import YeoJohnsonTransformer, PowerTransformer
from feature_engine.selection import SmartCorrelatedSelection

# Ordinal Encoder
binary_ordinal_encoder = OrdinalEncoder(
    encoding_method='arbitrary',
    variables=['hotel', 'is_repeated_guest']
)

# Month Mapper (before cyclical encoding)
month_mapper = MonthMapper(variables=['arrival_date_month'])

# Rare Label Encoders (before one hot encoding)
rare_country_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['country']
)
rare_agent_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['agent']
)
rare_company_encoder = RareLabelEncoder(
    tol=0.01,
    n_categories=1,
    variables=['company']
)

# One-Hot Encoder (after rare label encoders)
one_hot_encoder = OneHotEncoder(
    variables=[
        'meal', 'market_segment', 'distribution_channel', 'reserved_room_type',
        'assigned_room_type', 'deposit_type', 'customer_type', 'country', 'agent', 'company'],
    drop_last=False
)

# Cyclical Encoders (after Month Mapper)
cyclical_features = CyclicalFeatures(
    variables=['arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month'],
    drop_original=True
)

# Create New Features
create_total_stays = MathFeatures(
    variables=['stays_in_weekend_nights', 'stays_in_week_nights'],
    func='sum',
    new_variables_names=['total_stay_length']
)
create_total_cost = MathFeatures(
    variables=['adr', 'total_stay_length'],
    func='prod',
    new_variables_names=['total_cost']
)
create_cancellation_ratio = CancellationRatio()  # requires 'previous_cancellations' and 'previous_bookings_not_canceled'

# Numeric Transformers
adr_transformer = YeoJohnsonTransformer(variables=['adr'])
lead_time_transformer = PowerTransformer(variables=['lead_time'])

# Smart Correlated Selection
smart_corr_sel = SmartCorrelatedSelection(
    variables=None,
    method='spearman',
    threshold=0.6,
    selection_method='variance'
)

Define feature engineering pipeline

In [ ]:
from sklearn.pipeline import Pipeline

feature_engineering_pipeline = Pipeline([
    ('binary_ordinal_encoder', binary_ordinal_encoder),
    ('month_mapper', month_mapper),  # before cyclical encoding
    ('rare_country_encoder', rare_country_encoder),  # before one-hot encoding
    ('rare_agent_encoder', rare_agent_encoder),  # before one-hot encoding
    ('rare_company_encoder', rare_company_encoder),  # before one-hot encoding
    ('one_hot_encoder', one_hot_encoder),  # after rare-label encoding
    ('cyclical_features', cyclical_features),  # after month mapper
    ('create_total_stays', create_total_stays),  # before create_total_cost
    ('create_total_cost', create_total_cost),  # after create_total_stays
    ('create_cancellation_ratio', create_cancellation_ratio),
    ('adr_transformer', adr_transformer),
    ('lead_time_transformer', lead_time_transformer),
    ('smart_corr_sel', smart_corr_sel)
])

Fit the pipeline and transform data.

In [ ]:
X_train_copy = feature_engineering_pipeline.fit_transform(X_train_copy)
X_test_copy = feature_engineering_pipeline.transform(X_test_copy)

print('Dropped features:', smart_corr_sel.features_to_drop_)

print('X_train_copy:', X_train_copy.shape)
print('X_test_copy:', X_test_copy.shape)

X_train_copy.head(3)

# Conclusion and Next Steps


When testing different models, the following feature engineering transformers will be considered for including in the ML pipeline:
 
- OrdinalEncoder
  - `hotel`
  
- MonthMapper (a custom transformer):
  - `arrival_date_month`
  
- RareLabelEncoder:
  - `country`, `agent`, `company`
  
- OneHotEncoder
  - 'meal', 'market_segment', 'distribution_channel',  'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type'
  - (after RareLabelEncode) - `country`, `agent`, `company`
  
- CyclicalFeatures
  - `arrival_date_week_number`, `arrival_date_day_of_month`
  - (after mapping) `arrival_date_month`
  
- MathFeatures Encoding
  - total_stays (using 'stays_in_weekend_nights' and 'stays_in_week_nights')
  - total_cost (using 'adr' and 'total_stay_length')
  - cancel_ratio (using 'previous_cancellations' and 'previous_bookings_not_canceled')
  
- YeoJohnsonTransformer
  - `adr`
  
- PowerTransformer
  - `lead_time`
  
- SmartCorrelatedSelection
  - since it removes some of the new features created above, experiment with ignoring certain variables
 

***NOTE:*** *`reservation_status` and `reservation_status_date` were dropped as part of the data cleaning process.*

***NOTE:*** *Adding a flag for the inconsistency found between `is_repeated_guest` and `previous_bookings_not_canceled` will not be considered at this stage. It may be added later if the model is not performing well.*

Next we will train the classification model.